# Cat bond fund analysis

This analyzes a diverse range of cat bond funds with publicly available data.

## Sources

Data was gathered from various public sources. Links to sufficient sources are provided in the headers of the raw CSVs ([link](../data/raw/)) for reference and verification.

## Setup

### Imports

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from analysis import checks, plots
from analysis.data import CsvStore, ReturnProcessor
from analysis.figure import save
from analysis.paths import FIGURES_DIR, PROCESSED_DATA_DIR, RAW_DATA_DIR

raw = CsvStore(RAW_DATA_DIR)
processed = CsvStore(PROCESSED_DATA_DIR)
processor = ReturnProcessor(raw, processed)

## Check raw data

Before processing, compound the raw monthly returns within each calendar year and compare them with the reported annual returns. Both inputs are the unadjusted series from the raw CSVs. Only fund-years with all twelve months are included. The tolerance is derived from the quoting precision in `checks.TOL_PP`; larger gaps point to likely data errors. The plot is shown inline for spot-checking.

In [ ]:
def check_raw(tol: float = checks.TOL_PP) -> None:
    monthly = processor.load().returns
    yearly = processor.load_yearly()
    deviations = checks.annual_deviations(monthly, yearly)
    summary, flags = checks.summarize(deviations, tol=tol)

    print("Flags above tolerance (pp):")
    display(flags)
    print("Per-fund deviation summary (pp):")
    display(summary.round(3))

    plots.annual_deviations(deviations, tol=tol)


check_raw()

## Process returns

Build the named return panels used for analysis, and persist them.

In [ ]:
panels = processor.run(save=True)
print("Processed panels:\n- " + "\n- ".join(panels))

## Explore

In [ ]:
SELECTED_SERIES = (
    "Plenum CAT Bond UCITS Fund Index - Master Capital",
    "With Intelligence ILS Index - USD Hedged",
    "Fermat UCITS Cat Bond Fund",
    "Franklin Cat Bond UCITS Fund",
    "Leadenhall UCITS ILS Fund",
)

fee_normalized = panels["fee_normalized"]
cash_excess = panels["cash_excess"]

missing = [name for name in SELECTED_SERIES if name not in cash_excess.columns]
if missing:
    raise ValueError(f"Missing selected series: {missing}")

selected = list(SELECTED_SERIES)


def explore_returns() -> None:
    heatmap, _ = plots.monthly_heatmap(
        cash_excess,
        months=36,
        title="Monthly returns relative to USD cash",
    )
    save(heatmap, FIGURES_DIR / "heatmap.png")

    plots.growth(
        cash_excess,
        columns=selected,
        months=24,
        title="Cumulative performance relative to USD cash",
    )

    plots.drawdowns(
        fee_normalized,
        columns=selected,
        months=36,
    )

    plots.rolling_returns(
        cash_excess,
        columns=selected,
        window=12,
        display_months=36,
    )


explore_returns()


![heatmap](figures/heatmap.png)